# GPWv4: quickstart

Download GPWv4 population density at 2.5 arc-minute resolution, subset
to the UK, and plot a population density map.

**Prerequisites:**
- Free NASA Earthdata account: https://urs.earthdata.nasa.gov/users/new
- Credentials in `~/.netrc` (or `~/_netrc` on Windows) with an entry for
  `machine urs.earthdata.nasa.gov`
- `pip install requests rioxarray rasterio xarray matplotlib cartopy numpy`

See [`docs/gpw-population/README.md`](../docs/gpw-population/README.md)
for the full reference.

In [ ]:
# ==================================================================
# USER CONFIGURATION - Edit these values for your use case
# ==================================================================
YEAR = 2020
RESOLUTION = "2pt5_min"  # 30_sec, 2pt5_min, 15_min, 30_min, 1_deg

# UK bounding box
BBOX = {"north": 55, "south": 49, "west": -8, "east": 2}

OUTPUT_DIR = "../data/gpw"
# ==================================================================

## Imports and environment check

In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

print(f"Python       {sys.version.split()[0]}")
for pkg in ["requests", "rioxarray", "rasterio", "xarray", "matplotlib", "cartopy"]:
    print(f"{pkg:12} {version(pkg)}")

def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / "CLAUDE.md").exists():
            return parent
    raise RuntimeError("Could not find repo root.")

repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

## Download population density

Downloads from SEDAC with NASA Earthdata authentication. The 2.5
arc-minute GeoTIFF is a manageable size (tens of MB). The zip file is
downloaded and the GeoTIFF extracted automatically.

In [ ]:
from scripts.gpw_population_download import download  # noqa: E402

tif_path = download(year=YEAR, resolution=RESOLUTION, output_dir=OUTPUT_DIR)
print(f"GeoTIFF: {tif_path} ({tif_path.stat().st_size / 1e6:.2f} MB)")

## Open and subset to the UK

GPWv4 GeoTIFFs are read with `rioxarray`. The data has latitude
descending (north to south), so we slice y from north to south.

In [ ]:
import rioxarray  # noqa: F401 (registers the rio accessor)

da = rioxarray.open_rasterio(tif_path)
print(da)

# Subset to UK bounding box
uk = da.sel(
    x=slice(BBOX["west"], BBOX["east"]),
    y=slice(BBOX["north"], BBOX["south"]),  # y is descending
)

# Mask NoData values
nodata = da.attrs.get("_FillValue", -3.4028235e+38)
uk = uk.where(uk != nodata)

# Drop the band dimension (single band)
uk = uk.squeeze("band", drop=True)

print(f"\nUK subset shape: {uk.shape}")
print(f"Population density range: {float(uk.min()):.1f} to {float(uk.max()):.1f} persons/km2")

## Plot UK population density

Population density on a log scale to show both urban centres
and rural areas. London and the major cities stand out clearly.

In [ ]:
import matplotlib.colors as mcolors

fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.PlateCarree())

# Log scale to show the full range from rural to urban
pop_data = uk.values.copy()
pop_data[pop_data <= 0] = np.nan

im = ax.pcolormesh(
    uk.x.values, uk.y.values, pop_data,
    transform=ccrs.PlateCarree(),
    cmap="YlOrRd",
    norm=mcolors.LogNorm(vmin=1, vmax=10000),
)

ax.coastlines(resolution="50m", linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor="gray")
gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
gl.top_labels = False
gl.right_labels = False

cbar = plt.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label(f"Population density {YEAR} (persons/km2)")

ax.set_title(f"GPWv4 population density, UK, {YEAR}")
plt.tight_layout()
plt.show()

## Next steps

- **Full reference**: [`docs/gpw-population/README.md`](../docs/gpw-population/README.md)
- **Climate risk overlay**: combine this population grid with a climate
  hazard layer (e.g. ERA5 temperature extremes) to estimate exposed
  populations.
- **Higher resolution**: switch to `30_sec` for the native ~1 km grid
  (warning: several GB).
- **UN WPP-adjusted**: use the adjusted variant if you need country
  totals to match UN population estimates.
- **Dasymetric alternatives**: WorldPop and GHS-POP redistribute
  population using land cover and building footprints for finer spatial
  detail within administrative units.